<a href="https://colab.research.google.com/github/alysaqiib/flyrank_internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

My lane: Refresh / Content Opportunity Scoring. Why: the product team benefits from a ranked review queue that sends editors the pages most likely to be worth fixing. The starter dataset contains a reasonable number of pages with measurable demand, so we can produce a ranked list with reason codes and measure precision@K for an editorial workflow.

In [1]:
import pandas as pd
pd.options.display.max_columns = 200
df = pd.read_csv("content_refresh_anonymized.csv")
df.shape, df.head(3)

((30000, 44),
              content_id          client_id  search_volume  competition  \
 0  content_304f48230142  client_f369cb89fc           10.0         0.67   
 1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
 2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
 
   competition_level   cpc     content_type    main_intent  word_count  \
 0              HIGH  2.05  keyword article  transactional      3221.0   
 1               LOW  0.05  keyword article  informational      2481.0   
 2               LOW  0.00  keyword article  informational      3515.0   
 
    char_count provider_used              model_used  impressions_90d  \
 0     20457.0           NaN        gemini-2.5-flash             3803   
 1     15562.0           NaN  gemini-3-flash-preview            15320   
 2     23643.0           NaN        gemini-2.5-flash            12581   
 
    clicks_90d  pageviews_90d  sessions_90d  users_90d  engaged_sessions_90d  \
 0          

## 2. The question: decision, action, cost of a wrong call

Decision: Which pages should be reviewed first for refresh/expansion/protection.
        Who acts & action: A content reviewer/editor opens the page, inspects reason codes, and chooses one of: refresh content, monitor, merge, or ignore.
        Cost of a wrong recommendation: False positives waste editor time (estimated X–Y minutes per page). False negatives miss recovery opportunities and potential traffic/revenue. We will choose thresholds to balance reviewer capacity and precision.


In [2]:
n_rows = len(df)
n_clients = df['client_id'].nunique()
with_impr_frac = (df['impressions_90d'] > 0).mean()
print(f"rows: {n_rows:,}, unique clients: {n_clients:,}")
print(f"fraction with impressions_90d > 0: {with_impr_frac:.3f} ({with_impr_frac*100:.1f}%)")


rows: 30,000, unique clients: 32
fraction with impressions_90d > 0: 1.000 (100.0%)


## 3. Quick look at the data (2-3 real numbers)

 I loaded the starter CSV and produced a few summary numbers below that show this lane is viable

In [3]:
n_500 = (df['impressions_90d'] >= 500).sum()
n_100 = (df['impressions_90d'] >= 100).sum()
print(f"rows with impressions_90d >= 500: {n_500:,}")
print(f"rows with impressions_90d >= 100: {n_100:,}")

rows with impressions_90d >= 500: 16,726
rows with impressions_90d >= 100: 22,006


## 4. Careful words: what I can and can't claim

What I can claim: Observational associations and a ranked decision-support queue that prioritizes pages for manual review. I can show which signals correlate with the starter label and produce a transparent baseline and a model that improves precision@K on the starter slice.
What I can't claim: Causal claims that a refresh will cause recovery (no experiment). I can't reconstruct raw URLs, client names, or publish private data. The starter label is a current-window proxy (trend_direction); it's not a future-window ground truth. There are gotchas: rate columns are percent-like (ctr=0.76 means 0.76%), avg_position=0 means no data, and trend_pct/trend_direction relationships can leak information if used incorrectly.

In [4]:
if 'is_declining_label' in df.columns:
    print("is_declining_label distribution:")
    print(df['is_declining_label'].value_counts(normalize=True))
else:
    print("trend_direction distribution:")
    print(df['trend_direction'].value_counts(normalize=True))

# CTR gotcha and avg_position no-data
mean_ctr_raw = df['ctr'].dropna().mean()
n_avgpos0 = (df['avg_position'] == 0).sum()
print(f"mean raw ctr value: {mean_ctr_raw:.4f} (interpret as {mean_ctr_raw/100:.4f} fraction)")
print(f"avg_position == 0 rows (no-data): {n_avgpos0:,} ({n_avgpos0/len(df):.3%})")

trend_direction distribution:
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64
mean raw ctr value: 0.5107 (interpret as 0.0051 fraction)
avg_position == 0 rows (no-data): 1,205 (4.017%)


## Self-check

Before you submit, confirm each line honestly:

- [.] Every section above is filled — markdown thinking AND the code that backs it
- [.] The notebook runs top to bottom with no errors (Runtime → Run all)
- [.] No client names, URLs, or private queries anywhere
- [.] My claims use careful words: observed, measured, directional, decision-support
- [.] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.